In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl
import os

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.optimize import curve_fit

import matplotlib.lines as mlines

import skunk

plt.rcParams.update({'font.size': 14})

In [ ]:
import os
import io

def safe_svg2pdf(svg: str, output_path: str = None) -> bytes:
    """
    Converts SVG string to PDF and writes to output_path if provided.
    Returns PDF as bytes either way.

    Args:
        svg (str): SVG content.
        output_path (str, optional): Path to write PDF file. Can include ~.

    Returns:
        bytes: The resulting PDF file content.
    """
    buffer = io.BytesIO()

    try:
        # Write to in-memory buffer first
        cairosvg.svg2pdf(bytestring=svg, write_to=buffer)
        pdf_data = buffer.getvalue()

        if output_path:
            # Expand ~ and write manually to file
            expanded_path = os.path.expanduser(output_path)
            with open(expanded_path, 'wb') as f:
                f.write(pdf_data)
            print(f"✅ PDF written to: {expanded_path}")
        else:
            print("✅ PDF generated in memory.")

        return pdf_data

    except OSError as e:
        print(f"❌ CairoSVG failed: {e}")
        return b''  # Or raise if you'd prefer

import os
import io


def safe_svg2pdf(svg: str, output_path: str = None) -> bytes:
    """
    Converts SVG string to PDF and writes to output_path if provided.
    Returns PDF as bytes either way.

    Args:
        svg (str): SVG content.
        output_path (str, optional): Path to write PDF file. Can include ~.

    Returns:
        bytes: The resulting PDF file content.
    """
    buffer = io.BytesIO()

    try:
        # Write to in-memory buffer first
        cairosvg.svg2pdf(bytestring=svg, write_to=buffer)
        pdf_data = buffer.getvalue()

        if output_path:
            # Expand ~ and write manually to file
            expanded_path = os.path.expanduser(output_path)
            with open(expanded_path, 'wb') as f:
                f.write(pdf_data)
            print(f"✅ PDF written to: {expanded_path}")
        else:
            print("✅ PDF generated in memory.")

        return pdf_data

    except OSError as e:
        print(f"❌ CairoSVG failed: {e}")
        return b''  # Or raise if you'd prefer



In [ ]:
def load1dFig2(i):
    return np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD',  'data', f'{i}', 'data.tsv'))
def load2dFig2(i, num):
    tmp = np.loadtxt(os.path.join('..', '20230512 HMIA13-5 QD',  'data', f'{i}', 'data.tsv'))
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
# Gain table
q1 = 100.075
q2 = 100.103
q3 = 100.087
q4 = 100.082

s1 = 100.104
s2 = 99.977
s3 = 100.183

In [ ]:
def fitG(deltaVg, G0, x):
    return G0*((1e-6+deltaVg*x)/(1e-6+np.sinh(deltaVg*x)))

In [ ]:
def fitGauss(deltaVg, G0, x):
    return G0*np.exp(-(deltaVg*x)**2)

In [ ]:
fig, ax = plt.subplots()

x = 10
deltaVg = np.linspace(-1, 1, 101)
G0 = 1

ax.plot(deltaVg, fitG(deltaVg, G0, x))

popt, pcov = curve_fit(fitGauss, deltaVg, fitG(deltaVg, G0, x))
ax.plot(deltaVg, fitGauss(deltaVg, popt[0], popt[1]))

ax2 = ax.twinx()
ax2.plot(deltaVg, fitGauss(deltaVg, popt[0], popt[1]) - fitG(deltaVg, G0, x), 'r')

In [ ]:
# %%
# fig, ax = plt.subplots(2, 3, figsize=(10, 8),gridspec_kw={"width_ratios":[1,1, 0.001],"height_ratios":[1, 1]})
# ax1 = ax[0,0]
# ax2 = ax[0,1]
# ax3 = ax[1,0]
# ax4 = ax[1,1]
# figinset, inset_ax = plt.subplots()
fig2 = plt.figure(figsize=(16, 8),constrained_layout=True)

gs = fig2.add_gridspec(2, 4, width_ratios=(1,1,1,1))
# gs = GridSpec(3, 3, figure=fig1)
f2_ax1 = fig2.add_subplot(gs[:1, :2])
# f3_ax1.set_title('gs[0, :3]')
f2_ax2 = fig2.add_subplot(gs[:1, 2:4])
f2_ax4 = fig2.add_subplot(gs[1:2, 3])
# inset_ax = fig2.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')
f2_ax3 = fig2.add_subplot(gs[1:2,:3])

f2_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f2_ax1.transAxes)

f2_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f2_ax2.transAxes)
f2_ax1.set_axis_off()
# skunk.connect(f1_ax3, 'zoom')
# f2_ax3.set_axis_off()


# ax2=ax.twinx()
offset=0
npoints=[20]
colors = ['b', 'r' ,'k']
for i, d in enumerate([77]):
    dat = load1dFig2(d)
    vg = dat[:, 1]
    vplunger = vg
#     vs = dat[:, 2]*100/q1
    vt = dat[:, 4]*100/q2
    vr = dat[:, 2]*100/q1

    g = 3*vt/(vt+vr)
    G = g
    alpha = 0.16/5.9
    widths = np.zeros((58))
    width_height = np.zeros((58))
    left = np.zeros((58))
    right = np.zeros((58))
    start=2
    peaks, _ = find_peaks(G[start:], prominence=0.1)
    Gpeak = G[start+peaks]
    Gmax = np.max(G[start:])
    print(np.diff(vplunger[start + peaks]))
#         phase = dat[:, 3]
#         v = 5e-6
#         vqpc = -1.988 - 0.002*i
    widths[:len(peaks)], width_height[:len(peaks)], left[:len(peaks),], right[:len(peaks)] = peak_widths(G[start:], peaks, rel_height=0.5)
#         ax.plot(1*0.03*i+curr[730:930]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.plot(vplunger[start:], offset*0.005*i+G[start:], 'o-', lw=0.75, markersize=2, label=labels[i])
    
#         ax.plot(1*0.03*i+curr[left[i]]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.semilogy(vplunger[start+peaks],offset*0.005*i+G[start+peaks],'b*')

    for peaknum in range(0,12):
        Vg0 = vplunger[start+ peaks[peaknum]]
#         print(Vg0)
        G0 = np.max(G[start+peaks[peaknum]])
#         alpha = 0.04
        Vg = vplunger[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        deltaVg = Vg - Vg0
        Gfit = G[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        popt, pcov = curve_fit(fitG, deltaVg, Gfit, p0=[G0, 11000])
#             popt2, pcov2 = curve_fit(fitGdot, deltaVg, Gfit, p0=[G0, 5000])
        print(alpha*1*(popt[1]**-1)*1e6/86)
#         print(alpha*0.5*(popt2[1]**-1)*1e6/86)

#         print(popt[1])
        # ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'ro-', lw=1)#, label='SET')

    f2_ax2.plot(vplunger[:]+ 0.0*(i//2),0.0*i+ G[:], 'bx-', lw=1, markersize=2)
    # f2_ax2.legend(loc = 'center left')
    # f2_ax2.legend(loc = 'center left')

f2_ax2.grid(ls='--', lw=0.4)
f2_ax2.set_xlabel('V$_{pR}$ (V)')
f2_ax2.set_ylabel('G$(e^2/h)$')
f2_ax2.set_yscale('linear')
# ax[0].set_xlim(-3.5, -2.4)
f2_ax2.set_ylim(-0.01, 0.21)


# left, bottom, width, height = [1.0, 0.6, 0.2, 0.2]
# inset_ax = fig2.add_axes([left, bottom, width, height])

# inset_ax = inset_axes(f2_ax2,
#                     width="30%", # width = 30% of parent_bbox
#                     height=1., # height : 1 inch
#                     bbox_to_anchor=(0,0,1,1), bbox_transform=f2_ax2.transAxes,
#                     loc=1)
inset_ax = f2_ax4
# mpl.style.use('default')
# labels=['bottom', 'top', 'top']
# ax2=ax.twinx()
offset=0
npoints=[20]
colors = ['b', 'r' ,'k']
for i, d in enumerate([77]):
    dat = load1dFig2(d)
    vg = dat[:, 1]
    vplunger = vg
#     vs = dat[:, 2]*100/q1
    vt = dat[:, 4]*100/q2
    vr = dat[:, 2]*100/q1

    g = 3*vt/(vt+vr)
    G = g
    alpha = 0.16/5.9
    widths = np.zeros((58))
    width_height = np.zeros((58))
    left = np.zeros((58))
    right = np.zeros((58))
    start=2
    peaks, _ = find_peaks(G[start:], prominence=0.1)
    Gpeak = G[start+peaks]
    print('------------')
    print(np.mean(Gpeak))
    print(np.std(Gpeak))
    print('------------')
    Gmax = np.max(G[start:])
    print(np.diff(vplunger[start + peaks]))
#         phase = dat[:, 3]
#         v = 5e-6
#         vqpc = -1.988 - 0.002*i
    widths[:len(peaks)], width_height[:len(peaks)], left[:len(peaks),], right[:len(peaks)] = peak_widths(G[start:], peaks, rel_height=0.5)
#         ax.plot(1*0.03*i+curr[730:930]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.plot(vplunger[start:], offset*0.005*i+G[start:], 'o-', lw=0.75, markersize=2, label=labels[i])
    
#         ax.plot(1*0.03*i+curr[left[i]]/v*25813, lw=0.75, label=str(vqpc)+' V', color='k')
    # ax.semilogy(vplunger[start+peaks],offset*0.005*i+G[start+peaks],'b*')
    startpeak = 10
    endpeak = 11
    inset_ax.plot(vplunger[start + peaks[startpeak] - 2*npoints[i] : start + peaks[endpeak-1] + 2*npoints[i]], G[start + peaks[startpeak] - 2*npoints[i] : start + peaks[endpeak-1] + 2*npoints[i]], 'bx-', lw=1)#, label='SET')
    print(f'Number of poinste = {peaks[endpeak-1] - peaks[startpeak] + 4*npoints[i]}')
    Gfiterror = np.zeros((16))
    Gfitmean = np.zeros((16))
    weighted_sum = 0
    weight_norm = 0
    for peaknum in range(10,11):
        Vg0 = vplunger[start+ peaks[peaknum]]
#         print(Vg0)
        G0 = np.max(G[start+peaks[peaknum]])
#         alpha = 0.04
        Vg = vplunger[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        deltaVg = Vg - Vg0
        Gfit = G[start + peaks[peaknum] - npoints[i] : start + peaks[peaknum] + npoints[i]]
        popt, pcov = curve_fit(fitG, deltaVg, Gfit, p0=[G0, 11000])
        popt2, pcov2 = curve_fit(fitGauss, deltaVg, Gfit, p0=[G0, 11000/3])
#             popt2, pcov2 = curve_fit(fitGdot, deltaVg, Gfit, p0=[G0, 5000])
        print(alpha*1*(popt[1]**-1)*1e6/86)
        perr = np.sqrt(np.diag(pcov))
        perr2 = np.sqrt(np.diag(pcov2))
        print(alpha*1*(perr[1]*popt[1]**-2)*1e6/86)
        # print('------------')
        Gfitmean[peaknum] = popt[0]
        Gfiterror[peaknum] = perr[0]
        weighted_sum = weighted_sum + Gfitmean[peaknum]/(Gfiterror[peaknum]**2)
        weight_norm = weight_norm + 1/(Gfiterror[peaknum]**2)
        # print('------------')
#         print(alpha*0.5*(popt2[1]**-1)*1e6/86)

#         print(popt[1])
        if peaknum==startpeak:
            inset_ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'r--', lw=2, label=f'{1e3*alpha*1*(popt[1]**-1)*1e6/86:.2f} $\pm$ {1e3*alpha*1*(perr[1]*popt[1]**-2)*1e6/86:.2f} mK')
            # inset_ax.plot(Vg, 0*0.005*i+fitGauss(deltaVg, popt2[0], popt2[1]), 'k--', lw=2, label=f'{1e3*alpha*1*(popt2[1]**-1)*1e6/86:.2f} $\pm$ {1e3*alpha*1*(perr2[1]*popt2[1]**-2)*1e6/86:.2f} mK')
        else:
            inset_ax.plot(Vg, 0*0.005*i+fitG(deltaVg, popt[0], popt[1]), 'r--', lw=2)#, label='SET')

    # ax.plot(vplunger[:]+ 0.0*(i//2),0.0*i+ G[:], 'b*-')
    # inset_ax.legend()
    # inset_ax.legend(loc='lower right', framealpha=1)
print(f'Weighted average of peak heights = {weighted_sum/weight_norm:.3}')
print(f'Error in estimation of peak heights = {np.sqrt(1/weight_norm):.3}')
inset_ax.grid(ls='--', lw=0.4)
inset_ax.set_xlabel('V$_{pR}$ (V)')
inset_ax.set_ylabel('G$(e^2/h)$')
inset_ax.set_yscale('log')
# ax[0].set_xlim(-3.5, -2.4)
inset_ax.set_ylim(4e-3, 0.21)

f2_ax3.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f2_ax3.transAxes)
f2_ax4.text(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f2_ax4.transAxes)

dat = load2dFig2(79, 101)
plunger = dat['1']
idc = dat['2']
vr = dat['3']
vt = dat['5']
vrdc = dat['9']*1/100
vtdc = dat['10']/100
vsdc = vrdc + vtdc
dcoffset = np.mean(vsdc[50,:])
G = 3*vt/(vt+vr)


im = f2_ax3.pcolormesh(plunger, (vsdc - dcoffset)*1e6, np.abs(G),vmin=1e-3, vmax=0.2, cmap='magma', rasterized=True)



#fig, ax = plt.subplots()
#ax2 = ax.twinx()
Gsavgol = G
Gsavgol = savgol_filter(G, 10, 4, deriv=0, delta=np.diff(plunger[0,:])[0], axis=1) 
# im = ax.pcolormesh(plunger, (vsdc - dcoffset)*1e6, np.abs(Gsavgol),vmin=1e-3, vmax=0.2, cmap='magma', rasterized=True)
#start=10
#stop=100



peaks = np.zeros((8, G.shape[0]))
vspeak = np.zeros(G.shape[0])
#ax.plot(plunger[0,:], Gsavgol[50, :], 'k', lw=0.5)
zbpeak, _ = find_peaks(Gsavgol[50, :], prominence=0.03)
ax.plot(plunger[0,zbpeak], Gsavgol[50, zbpeak], 'ro')

f2_ax3.plot(plunger[0,zbpeak], 1e6*(vsdc[50,zbpeak]- dcoffset), 'ro')
print(1e6*(vsdc[50,zbpeak]- dcoffset))

positive_slopes = np.array([0.050142, 0.050142, 0.050142, 0.050142,
                            0.050142, 0.050142, 0.050142, 0.050142,
                            0.050142, 0.050142, 0.050142, 0.050142,
                            0.050142, 0.050142, 0.050142, 0.050142,
                            0.050142, 0.050142, 0.050142, 0.050142])
#f2_ax3.plot(plunger[0,zbpeak[12]], 1e6*(vsdc[50,zbpeak[12]]-dcoffset), 'bo')

adjustments  = np.array([ 1.10, 1.120, 1.070, 1.0200,
                         1.0000, 1.1300, 1.1000, 1.1400,
                         1.1100, 1.330, 1.100, 1.140,
                         1.1000, 1.1000, 1.1500, 1.1000,
                         1.0000, 1.0500, 1.0700, 1.0700])

negative_slopes = -0.15841*np.ones_like(positive_slopes)

adjustments_neg  = np.array([ 1.10, 1.120, 1.070, 1.0200,
                         1.0000, 1.1300, 1.1000, 1.1400,
                         1.1100, 1.330, 1.100, 1.140,
                         1.1000, 1.1000, 1.1500, 1.1000,
                         1.0000, 1.0500, 1.0700, 1.0700])

positive_slopes = positive_slopes*adjustments
negative_slopes = negative_slopes*adjustments_neg

intercepts_pos = (vsdc[50,zbpeak]- dcoffset)-positive_slopes*plunger[0,zbpeak]
intercepts_neg = (vsdc[50,zbpeak]- dcoffset)-negative_slopes*plunger[0,zbpeak]

isectx_top = -(np.roll(intercepts_neg,-1)[:-1]- intercepts_pos[:-1])/(np.roll(negative_slopes, -1)[:-1]- positive_slopes[:-1])
isecty_top = (positive_slopes[:-1]*isectx_top + intercepts_pos[:-1])


mask = np.ones_like(isectx_top, dtype=bool)
exclude = np.array([3, 4, 16])
mask[exclude] = False

exclude_color = 'black'
f2_ax3.plot(isectx_top, 1e6*isecty_top, 'bo')
f2_ax3.plot(isectx_top[exclude], 1e6*isecty_top[exclude], marker = 'o', linestyle = 'None', color = exclude_color)

isectx_bot = -(intercepts_neg[:-1]- np.roll(intercepts_pos, -1)[:-1])/(negative_slopes[:-1]- np.roll(positive_slopes,-1)[:-1])
isecty_bot = (negative_slopes[:-1]*isectx_bot + intercepts_neg[:-1])



f2_ax3.plot(isectx_bot, 1e6*isecty_bot, 'bo')
f2_ax3.plot(isectx_bot[exclude], 1e6*isecty_bot[exclude], marker = 'o', linestyle = 'None', color = exclude_color)

charge_energy = 0.25*1e6*(isecty_top-isecty_bot)[mask]
print(f'Mean charge energy = {np.mean(charge_energy):.2f} +/- {np.std(charge_energy)/np.sqrt(len(charge_energy)):.2f} ueV')

for k in range(len(zbpeak)):
    xvals =np.linspace(plunger[0,zbpeak[k]] - 0.005, plunger[0,zbpeak[k]] + 0.005, 100)
    yvals_pos = positive_slopes[k] * xvals + intercepts_pos[k]
    yvals_neg = negative_slopes[k] * xvals + intercepts_neg[k]
    f2_ax3.plot(xvals, 1e6*yvals_pos, 'k--')
    f2_ax3.plot(xvals, 1e6*yvals_neg, 'w--')




f2_ax3.set_xlabel('$V_{pR}$ (V)')
f2_ax3.set_ylabel('$V_{dc}$ ($\mu$V)') 
f2_ax3.set_ylim(-240, 240)
f2_ax3.set_xlim(np.amin(plunger), -0.44)
cax = f2_ax3.inset_axes([1.01, 0, 0.02, 1])
cbar = fig2.colorbar(im, cax=cax, extend='max')
cbar.ax.set_ylabel('$G (e^2/h)$')


skunk.connect(f2_ax1, 'sk2')

# svg = skunk.pltsvg(fig=fig2)
svg = skunk.insert(
    {  
       'sk2': 'fig3a.svg'
        #'sk2': '/Users/karnamorey/Library/CloudStorage/GoogleDrive-kmorey@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/ChargeNoiseFig4apaper.svg'        
    })

# fig2.set_constrained_layout(False)
# fig2.tight_layout()

# fig1.canvas.draw()
# # we want the legend included in the bbox_inches='tight' calcs.
# # cbar1.set_in_layout(True)
# # cbar2.set_in_layout(True)
# # we don't want the layout to change at this point.
#fig1.tight_layout()
    
skunk.display(svg)

# cairosvg.svg2pdf(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure2font14.pdf')
# cairosvg.svg2eps(bytestring=svg, write_to='/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/QDots-2/Papers/Hybrid Dot APL/figure2font14.eps')

# fig1.savefig('Figure1.pdf', dpi=300)
# ax[0,2].axis('off')
# ax[1,2].axis('off')
# plt.tight_layout()

# plt.savefig("Figure4.eps")